# Geothermal DOE GDB: GCS → Linux inventory (Colab)

**Use [Colab in the browser](https://colab.research.google.com/)** if the VS Code / Cursor Colab extension shows websocket or kernel errors.

**Windows `Permission denied` on `.gdb` rasters:** run this notebook on Colab (Linux).

**Workflow (recommended):** **Git** = source of truth for `scripts/` (clone/update into `/content/GeothermalAI`). **GCS** = large GIS data (`gsutil rsync` into `/content/...`). Same Google account for Colab + GCP. Push from your laptop; pull/clone in Colab each session.

**Alternatives:** upload `scripts/` to `/content/scripts`, or mount Drive — see optional section below.

See `colab/TROUBLESHOOTING_VSCODE_COLAB.md` if the Colab extension misbehaves.

### Your Windows PC: `gcloud` and `gsutil` (optional)

Use this when you run **`gsutil`** from **PowerShell** on your laptop (for example syncing large folders to your bucket before opening this notebook in Colab).

1. Install the [Google Cloud CLI](https://cloud.google.com/sdk/docs/install) — it includes **both** `gcloud` and `gsutil`.
2. **Restart Cursor** or open a **new** terminal so **PATH** includes `...\google-cloud-sdk\bin` (otherwise `gcloud` may be “not recognized”).
3. Confirm tools work:
   - `gcloud --version`
   - `gsutil version`
4. When you use this project on a machine for the first time (or after a long break):
   - `gcloud auth login`
   - `gcloud config set project YOUR_PROJECT_ID`  
   Use the **Project ID** from [Google Cloud Console](https://console.cloud.google.com/) (lowercase id), not the display name.

**Colab** uses the notebook’s in-runtime Google sign-in (`auth.authenticate_user`, etc.) — that is **separate** from your PC. Still keep **`GCP_PROJECT`** in the config cell aligned with the same project.

## 1) Geospatial stack (no kernel restart)

Uses **micromamba** into `/content/geo-env` so **Cursor/VS Code** does not hit a **condacolab kernel restart** (that often crashes the remote bridge).

First run ~5–15 minutes. Re-run only if you reset the runtime.

In [ ]:
%%bash
set -e
cd /content
ARCH="linux-64"
MM="/content/bin/micromamba"
mkdir -p /content/bin
if [ ! -x "$MM" ]; then
  TMPDIR=$(mktemp -d)
  cd "$TMPDIR"
  curl -Ls "https://micro.mamba.pm/api/micromamba/${ARCH}/latest" | tar -xvj bin/micromamba
  cp bin/micromamba "$MM"
  chmod +x "$MM"
  cd /content
  rm -rf "$TMPDIR"
fi
export MAMBA_ROOT_PREFIX=/content/mamba
"$MM" create -y -p /content/geo-env -c conda-forge python=3.11 gdal fiona rasterio numpy

conda-forge/linux-64                                        Using cache
conda-forge/noarch                                          Using cache


Transaction

  Prefix: /content/geo-env

  Updating specs:

   - python=3.11
   - gdal
   - fiona
   - rasterio
   - numpy


  Package                  Version  Build                 Channel           Size
──────────────────────────────────────────────────────────────────────────────────
  Install:
──────────────────────────────────────────────────────────────────────────────────

  + _openmp_mutex              4.5  20_gnu                conda-forge     Cached
  + affine                   2.4.0  pyhd8ed1ab_1          conda-forge     Cached
  + attrs                   26.1.0  pyhcf101f3_0          conda-forge     Cached
  + blosc                   1.21.6  he440d0b_1            conda-forge     Cached
  + bzip2                    1.0.8  hda65f42_9            conda-forge     Cached
  + c-ares                  1.34.6  hb03c661_0            conda-f

In [ ]:
import subprocess

GEO_PYTHON = "/content/geo-env/bin/python"
out = subprocess.check_output(
    [
        GEO_PYTHON,
        "-c",
        "from osgeo import gdal; import fiona; print('GDAL', gdal.VersionInfo('RELEASE_NAME'))",
    ],
    text=True,
)
print(out.strip())

GDAL 3.12.2


## 2) Config: GCS, Git repo, local paths

**GCP project ID** must match Cloud Console (lowercase), e.g. `maloney-geog-473` — not the display name.

**Git:** set `GIT_REPO_URL` to your fork if needed. Private repos need a [token](https://github.com/settings/tokens) in the URL (use Colab secrets) — public `GarretMaloney/GeothermalAI` works as-is.

In [ ]:
import os

# --- Google Cloud Storage (large GIS data) ---
GCS_BUCKET = "gis-final-project"
GCS_PREFIX = "GIS Final Project"  # object prefix; spaces OK (rsync uses subprocess, not shell)
GCP_PROJECT = "maloney-geog-473"

LOCAL_SYNC_ROOT = "/content/GIS Final Project"
LOCAL_GDB_ROOT = "/content/GIS Final Project/1303/DOE_GDB"

# --- Git (Python scripts; edit if you use a fork) ---
GIT_REPO_URL = "https://github.com/GarretMaloney/GeothermalAI.git"
GIT_BRANCH = "main"
REPO_DIR = "/content/GeothermalAI"
SCRIPTS_DIR = os.path.join(REPO_DIR, "scripts")

GEO_PYTHON = "/content/geo-env/bin/python"

## 3) Clone or update repository

Fetches **`scripts/`** from **`GIT_REPO_URL`** into **`REPO_DIR`**. On a new Colab runtime this is a shallow `git clone`; if you re-run, it `git pull`s.

In [ ]:
import os
import shutil
import subprocess

# Clone on a fresh runtime; pull if the repo is already there (e.g. re-run notebook).
_git = os.path.join(REPO_DIR, ".git")
if os.path.isdir(_git):
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--ff-only"], cwd=REPO_DIR)
    print("Git pull OK:", REPO_DIR)
else:
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    subprocess.check_call(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            GIT_BRANCH,
            GIT_REPO_URL,
            REPO_DIR,
        ]
    )
    print("Cloned:", REPO_DIR)

_inv = os.path.join(SCRIPTS_DIR, "inventory_gdbs.py")
if not os.path.isfile(_inv):
    top = sorted(os.listdir(REPO_DIR)) if os.path.isdir(REPO_DIR) else []
    scr = sorted(os.listdir(SCRIPTS_DIR)) if os.path.isdir(SCRIPTS_DIR) else []
    raise FileNotFoundError(
        f"Missing {_inv}\n"
        f"  Repo root ({REPO_DIR}): {top!r}\n"
        f"  scripts/ dir: {scr!r}\n"
        "  Usually this means `scripts/` was never pushed to GitHub. "
        "On your PC: git add scripts && git commit && git push. "
        "In Colab after a bad clone: delete /content/GeothermalAI (Files app) and re-run this cell."
    )
print("SCRIPTS_DIR:", SCRIPTS_DIR)

Git pull OK: /content/GeothermalAI
SCRIPTS_DIR: /content/GeothermalAI/scripts


## 4) Authenticate and sync from GCS

In [ ]:
from google.colab import auth
import os
import subprocess

auth.authenticate_user()

subprocess.check_call(["gcloud", "config", "set", "project", GCP_PROJECT])

os.makedirs(LOCAL_SYNC_ROOT, exist_ok=True)
# Use subprocess (not !shell) so paths with spaces in GCS_PREFIX are one argument each.
_prefix = GCS_PREFIX.strip("/")
src = f"gs://{GCS_BUCKET}/{_prefix}/"
subprocess.check_call(["gsutil", "-m", "rsync", "-r", src, LOCAL_SYNC_ROOT])
print("Synced", src, "->", LOCAL_SYNC_ROOT)

Synced gs://gis-final-project/GIS Final Project/ -> /content/GIS Final Project


## 5) Optional: mount Drive

**Skip this** if you ran the **Git clone** cell above (default workflow).

Use only if your scripts live on Drive: set `SCRIPTS_DIR = "/content/drive/MyDrive/.../scripts"` after mounting, then run inventory/export.

In [ ]:
# Optional — skip if you already cloned the repo in §3.
from google.colab import drive

drive.mount("/content/drive")

## 6) Run inventory → JSON

In [ ]:
import os
import subprocess

out_json = "/content/inventory_1303_full.json"
py = GEO_PYTHON if os.path.isfile(GEO_PYTHON) else __import__("sys").executable
cmd = [
    py,
    f"{SCRIPTS_DIR}/inventory_gdbs.py",
    "--root", LOCAL_GDB_ROOT,
    "-o", out_json,
]
print(" ".join(cmd))
subprocess.check_call(cmd, cwd=SCRIPTS_DIR)
print("Wrote", out_json)

/content/geo-env/bin/python /content/GeothermalAI/scripts/inventory_gdbs.py --root /content/GIS Final Project/1303/DOE_GDB -o /content/inventory_1303_full.json
Wrote /content/inventory_1303_full.json


## 7) (Optional) Export rasters → GeoTIFF, upload to GCS

In [ ]:
import os
import subprocess
import sys

tif_dir = "/content/geotiff_export"
py = GEO_PYTHON if os.path.isfile(GEO_PYTHON) else sys.executable
cmd = [
    py,
    f"{SCRIPTS_DIR}/export_gdb_rasters_to_geotiff.py",
    "--inventory", "/content/inventory_1303_full.json",
    "--out-dir", tif_dir,
]
subprocess.check_call(cmd, cwd=SCRIPTS_DIR)
dst = f"gs://{GCS_BUCKET}/geotiff-export/"
subprocess.check_call(["gsutil", "-m", "rsync", "-r", tif_dir, dst])
print("Uploaded to", dst)

Uploaded to gs://gis-final-project/geotiff-export/


In [ ]:
from google.colab import files

files.download("/content/inventory_1303_full.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>